# CS383 — Assignment 2
## EDA and Visualization Report: NYC 311 Resolution Times

*Due: see LMS*

### Scenario
Your local Community Board office has asked you, as a data intern, to look into how long 311
complaints take to get resolved across the city. They need two things from you:

1. The supporting statistical analysis (distribution, outliers, borough comparison, correlation).
2. **One** short, plain-language write-up of a single finding that a board member — someone who has
   never opened a notebook or a spreadsheet — could read in thirty seconds and actually understand.

This is exactly the skill Lecture 5 built toward: doing the technical work, then translating it.

---

### Academic integrity and using AI

Same policy as Assignment 1. This assignment is meant to show **your own** understanding of
descriptive statistics, outlier detection, and — especially — your own judgment about what's worth
saying and how to say it in plain language. That judgment is the actual skill being assessed here,
and it's not something an AI tool should be doing for you.

**Allowed** — using an AI tool (Claude, ChatGPT, GitHub Copilot, etc.) as a *tutor*: asking it to
explain a concept, explain an error message, explain what a piece of syntax does, or point you toward
the right Pandas/Matplotlib/Seaborn method to look up.

**Not allowed** — pasting the assignment questions into an AI tool and submitting what it gives back;
having AI write your code for you; having AI write your reasoning or your Deliverable 5 report for
you. A plain-language summary written by an AI tool defeats the entire point of the assignment.

A good test: if you can't explain — out loud, without looking anything up — exactly what every line of
your submitted notebook does and why you made each judgment call, that's a sign you leaned on a tool
too much.

---

### Getting started

Run the cell below once. It loads the same NYC 311 snapshot used in Lecture 5 (or realistic sample
data if the shared file isn't available). **Do not skip or modify this cell** — every deliverable
below assumes `complaints_df` exists with these columns.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(10000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 10000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    agency_list = ["NYPD", "HPD", "DOB", "DSNY", "DOT", "DOHMH"]
    agency_p = [0.65, 0.15, 0.05, 0.05, 0.05, 0.05]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_weight = np.where(days.dayofweek >= 5, 0.6, 1.0)
    day_weight = day_weight / day_weight.sum()

    day_idx = rng.choice(n_days, size=n, p=day_weight)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")

    agency = rng.choice(agency_list, size=n, p=agency_p)
    is_nypd_fallback = agency == "NYPD"

    still_open = rng.random(n) < 0.15
    # NYPD dispatches and closes tickets same-day; other agencies (inspections,
    # scheduled follow-up) take much longer -- mirrors the real signal in the
    # live snapshot, so Deliverable 4 has something real to find offline too.
    resolution_hours = np.where(
        is_nypd_fallback,
        rng.gamma(shape=1.5, scale=0.8, size=n),
        rng.gamma(shape=2.0, scale=60.0, size=n),
    )
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "agency": agency,
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df.head()

---

### Explore before you answer

Before jumping into the required questions, take a few minutes to look around:

- What does `complaints_df.describe()` show you about `resolution_time_hours`?
- How many complaints are still open (no `closed_date`) and therefore have no resolution time?
- Which boroughs and complaint types show up most often?

Use the scratch cell below (and add more cells if you want) to explore. This part isn't graded
directly, but it'll make the required questions much easier.

In [ ]:
# Scratch space -- explore complaints_df here.


---

## Required Deliverable 1 — Distribution of Resolution Times (15 pts)

Compute the **mean**, **median**, and **standard deviation** of `resolution_time_hours` (drop missing
values first — those are complaints that are still open). Then make one histogram showing the shape of
the distribution.

In 1-2 sentences: is the distribution symmetric or skewed? What does that imply about whether the mean
or the median is the more honest way to describe a "typical" complaint's resolution time?

In [ ]:
# Your code here.


**Your answer (symmetric or skewed, mean vs. median):**



---

## Required Deliverable 2 — Flagging Outliers with the IQR Rule (15 pts)

Using the same 1.5×IQR rule from lecture, flag which resolution times count as outliers. Report the
IQR bounds and how many complaints get flagged.

**Before you write the code**, answer: do you think these flagged cases are more likely data-entry
errors, or real complaints that genuinely took an unusually long time? Justify your answer in 1-2
sentences.

**Your reasoning (before you code):**



In [ ]:
# Your code here.


---

## Required Deliverable 3 — Comparing Boroughs (15 pts)

Using a box plot, compare `resolution_time_hours` across the five boroughs. Which borough has the
**highest** median resolution time, and which has the **lowest**? Report the median for each, and
state the size of the gap between them in hours.

Note: a handful of rows have `borough` set to `"Unspecified"` rather than a real borough — filter
those out first, the same way you'd drop any row that doesn't answer the question being asked.

In [ ]:
# Your code here.


**Your answer (slowest / fastest borough, size of the gap):**



---

## Required Deliverable 4 — Correlation and a Causation Check (15 pts)

Create a new column `is_nypd` that is `1` when `agency == "NYPD"` and `0` otherwise. Compute the
correlation between `is_nypd` and `resolution_time_hours`.

This one comes out meaningfully different from zero — not just noise. Does that mean NYPD is simply
"better" or "faster" at its job than other agencies? Name **one plausible confounding variable** that
could explain most of this relationship without NYPD's efficiency directly causing it — the same kind
of check done for boroughs in Lecture 5's ice-cream/drowning example.

Note: some agencies only appear a handful of times in this data. An agency's median resolution time
isn't trustworthy if it's based on only a few complaints — the same problem as the `"Unspecified"`
borough in Deliverable 3. When you rank agencies, keep only agencies with at least 30 resolved
complaints.

In [ ]:
# Your code here.


**Your answer (correlation strength + one plausible confounder):**



---

## Required Deliverable 5 — The Non-Technical Report (30 pts)

Pick **one** finding from Deliverables 1-4 (or something else you noticed while exploring) that an
actual Community Board member — not a data scientist — would care about.

Using the four-part framework from lecture, write it up:

1. **Lead with the finding** — the one sentence a busy reader would actually remember.
2. **One supporting chart** — the smallest chart that proves the finding, not every chart you made.
3. **No jargon** — "IQR," "correlation," "standard deviation," and "outlier" mean nothing to this
   reader. Say what you found in plain language instead.
4. **State the "so what"** — why should this reader care? What would they do differently knowing this?

In [ ]:
# Your code here.


**Your report (plain language):**



---

## Grading Rubric

| Criterion | Points |
|---|---|
| Deliverable 1 — stats computed correctly; histogram shown; skew/mean-vs-median answer is correct | 15 |
| Deliverable 2 — IQR bounds and outlier count correct; reasoning is stated and defensible | 15 |
| Deliverable 3 — box plot correct; slowest/fastest borough and gap correctly identified | 15 |
| Deliverable 4 — correlation computed correctly; confounder is plausible and genuinely independent | 15 |
| Deliverable 5 — finding is genuinely non-technical, chart is the *right* one, "so what" is present | 30 |
| Notebook runs top to bottom without errors | 10 |
| **Total** | **100** |

---

## Submission

### GitHub (same repo as Assignment 1)

1. Use the same `cs383-assignments` repo you created for Assignment 1. Inside it, create a new folder:
   `assignment2/`.
2. Put your completed notebook in that folder, keeping its original filename:
   `assignment2_311_eda_report.ipynb`.

```bash
git add -A
git commit -m "Assignment 2: 311 EDA report"
git push
```

Submit the **direct link to your notebook file on GitHub** (the URL should end in
`assignment2/assignment2_311_eda_report.ipynb`) on BrightSpace under **Assignment 2**.

Make sure your notebook runs top to bottom without errors before you submit — a notebook that only
works because cells were run out of order will not be graded as passing.